In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv("data/sample_transactions.csv")

print(df.head())
print(df.info())
print(df.shape)
print(df.columns)

df["Date"] = pd.to_datetime(df["Date"])

On the above cell, imported libraries for the project, read the file, identified headers, then converted the dates.

WHY IS IT IMPORTANT TO CONVERT DATES? Pandas library initially reads date existing in csv files as string datatypes. Hence, the conversion turns the plain text into an actual date/time datatype that Pandas library understands.

ANALYSIS OF TRANSACTION TYPES IN THE SAMPLE DATA - This stage separates purchases, incoming payments, transfers, and debt payments.

In [ ]:
df["Description"].head(20)

This column is created to fulfill the purpose of the program to understand the difference between different transactions

In [ ]:
conditions = [
    df["Description"].str.contains(
        "Zelle Payment From",
        case=False,
        na=False
    ),
    df["Description"].str.contains(
        "Zelle Payment To", 
        case=False,
        na=False
    ),
    df["Description"].str.contains(
        "Capital One|Discover E-Payment",
        case=False,
        na=False
    ),
    df["Description"].str.contains(
        "Payment Sent.*Chime",
        case=False,
        na=False
    )
]

Let's look through what we did on the above cell:
"Zelle Payment" identification; what does case=False do? It ignores Capitalization
What does na=False do? It checks if the Zelle payment exists and returns a boolean value from every line
The same approach is used to identify sample debt-payment descriptions from different card providers.

In [ ]:
choices = [
    "Money In",
    "Transfer Out",
    "Debt Payment",
    "Transfer Out"
]

Condition             |Choice
Zelle Payment From    ->  Money In|
Zelle Payment To      ->  Transfer Out|
Capital One/Discover  ->  Debt Payment|

In [ ]:
df["Transaction_Type"] = np.select(
    conditions,
    choices, 
    default = "Purchase"
)

If the transaction doesn't match the assigned conditions so far, NumPy assumes it's a normal purchase

In [ ]:
df[
    ["Date", "Description", "Amount", "Transaction_Type"]
    ].head(25)

In [ ]:
df["Transaction_Type"].value_counts()

In [ ]:
df.groupby("Transaction_Type")["Amount"].sum()

In [ ]:
df[
    ["Description", "Amount", "Transaction_Type"]
    ]

So far we've turned messy transaction descriptions into meaningful financial transactions types

Next Stage is to Clean the Merchant Names

In [ ]:
df["Description"].head(20) #Inspecting my descriptions

In [ ]:
df["Description"].tail(20)

In [ ]:
df["Merchant"] = "Other" #Created an Empty Merchant Column

Until I recognize the merchant, call it other

In [ ]:
df[["Description", "Merchant"]].head()

Detecting Uber

In [ ]:
df.loc[
    df["Description"].str.contains("Uber", case=False, na=False),
    "Merchant"
    ] = "Uber"
#df["Description'].str.contain, finds every row containing the desired merchant
#df.loc[condition, "Merchant"], finds the rows matching the condition and modify the merchant column

In [ ]:
df[df["Merchant"] == "Uber"][
    ["Description", "Merchant"]
]
#Testing the above condition

All the ugly Uber descriptions have now been converted into Just "Uber" now

In [ ]:
df.loc[
    df["Description"].str.contains("DoorDash", case=False, na=False),
    "Merchant"
    ] = "DoorDash"

df[df["Merchant"] == "DoorDash"][
    ["Description", "Merchant"]
]

In [ ]:

df.loc[
    df["Description"].str.contains("Amazon", case=False, na=False),
    "Merchant"
] = "Amazon"
df.loc[
    df["Description"].str.contains("United", case=False, na=False),
    "Merchant"
] = "United Airlines"
df.loc[
    df["Description"].str.contains("Wm Supercenter", case=False, na=False),
    "Merchant"
] = "Walmart"

df.loc[
    df["Description"].str.contains("Autozone", case=False, na=False),
    "Merchant"
] = "AutoZone"

df.loc[
    df["Description"].str.contains("Raising Canes", case=False, na=False),
    "Merchant"
] = "Raising Cane's"
df.loc[
    df["Description"].str.contains("Fandango", case=False, na=False),
    "Merchant"
] = "Fandango"


Added all the merchants identified from the stament

In [ ]:
df.loc[
    df["Description"].str.contains("Capital One", case=False, na=False),
    "Merchant"
] = "Capital One"
df.loc[
    df["Description"].str.contains("Discover", case=False, na=False),
    "Merchant"
] = "Discover"
df.loc[
    df["Description"].str.contains("Chime", case=False, na=False),
    "Merchant"
] = "Chime"
df.loc[
    df["Description"].str.contains("Zelle", case=False, na=False),
    "Merchant"
] = "Zelle"

These are classified columns from stage two so now, Merchant = Zelle, Transaction_Type = Money In

In [ ]:
df["Merchant"].value_counts()

In [ ]:
df[df["Merchant"] == "Other"][
    ["Description", "Amount"]
]

In [ ]:
df.loc[
    df["Description"].str.contains("Pj's Coffee", case=False, na=False),
    "Merchant"
] = "PJ's Coffee"

In [ ]:
df[df["Merchant"] == "Other"][
    ["Description", "Amount"]
]

In [ ]:
df.loc[
    df["Description"].str.contains("Affirm.Com", case=False, na=False),
    "Merchant"
] = "Affirm Payment"

df.loc[
    df["Description"].str.contains("Lambright", case=False, na=False),
    "Merchant"
] = "Lambright LaTech Gym"


In [ ]:
df[df["Merchant"] == "Other"][
    ["Description", "Amount"]
]

In [ ]:
df.loc[
    df["Description"].str.contains("Affirm * Pay", case=False, na=False, regex=False),
    "Merchant"
] = "Affirm Purchase"
Subscription = [
    "Prime Video Channels",
    "Rocket Money Premium",
    "CircleK",
    "Apple",
    "LA Farm Bureau",
    "Mint Mobile",
    "Rocketfast Car Was"
    "Spotify",
]
pattern_subs = "|".join(Subscription)

df.loc[
    df["Description"].str.contains(pattern_subs, case=False, na=False),
    "Merchant"
] = "Subcription"

In [ ]:
df[df["Merchant"] == "Other"][
    ["Description", "Amount"]
]

In [ ]:
GasStations = [
    "Exxon",
    "Shell Oil",
    "Chevron",
    "Circle K",
    "QuickTrip",
    "Liquor Store",
]
pattern_gasst = "|".join(GasStations)

df.loc[
    df["Description"].str.contains(
        "Gazebo Smoke Shop",
        case=False,
        na=False
    ),
    "Merchant"
] = "Gazebo Smoke Shop"

df.loc[
    df["Description"].str.contains(pattern_gasst, case=False, na=False),
    "Merchant"
] = "Gas Station"

In [ ]:
df["Merchant"].value_counts()

In [ ]:
(
    df[df["Transaction_Type"] == "Purchase"]
    .groupby("Merchant")["Amount"]
    .sum()
    .abs()
    .sort_values(ascending=False)
)

In [ ]:
# Fix merchant names that were previously merged into categories

merchant_corrections = {
    
    # Gas station / convenience merchants
    # Delta goes first so "Exxon Delta Mini Mart" gets overwritten as Exxon afterward
    "Delta Mini Mart": r"Delta Mini Mart",
    "Exxon": r"Exxon",
    "Shell": r"Shell Oil",
    "Chevron": r"Chevron",
    "CircleK": r"Circlek",
    "QT": r"\bQT\b",

    # Subscription / recurring merchants
    "Spotify": r"Spotify",
    "Prime Video Channels": r"Prime Video Channels",
    "Rocket Money": r"Rocket Money Premium",
    "Apple": r"Apple\.Com",
    "LA Farm Bureau": r"LA Farm Bureau",
    "Mint Mobile": r"Mint Mobile",
    "Rocketfast Car Wash": r"Rocketfast",

    # Financing
    "Affirm": r"Affirm"
}


# Go through every merchant rule and update matching rows
for merchant, pattern in merchant_corrections.items():
    
    df.loc[
        df["Description"].str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        ),
        "Merchant"
    ] = merchant


# Check whether any old merged labels are still left
print("Still incorrectly merged:")

display(
    df[df["Merchant"].isin(["Gas Station", "Subscription"])][
        ["Description", "Merchant", "Amount"]
    ]
)


# Show updated spending totals by merchant
print("\nUpdated Merchant Totals:")

display(
    df[df["Transaction_Type"] == "Purchase"]
    .groupby("Merchant")["Amount"]
    .sum()
    .abs()
    .sort_values(ascending=False)
)

So so far I made a mistake by merging merchants as an category, turns out I needed to make a different column called category and the move from there. I've made the correction and now my updated DataFrame looks like this. Now we move to stage 4.

Building a category Dictionary

In [ ]:
category_map = {

    #Travel
    "United Airlines": "Travel",

    #Food
    "DoorDash": "Food Delivery",
    "PJ's Coffee": "Dining",
    "Raising Cane's": "Dining",

    #Transportation
    "Uber": "Transportation",

    #Gas / Convenience Stores
    "Exxon": "Gas/Convenience",
    "Delta Mini Mart": "Gas/Convenience",
    "QT" : "Gas/Convenience",
    "Shell": "Gas/Convenience",
    "CircleK": "Gas/Convenience",
    "Chevron": "Gas/Convenience",

    #Shopping
    "Amazon": "Shopping",
    "Walmart": "Groceries",

    #Car
    "Autozone": "Car",
    "Rocketfast Car Wash": "Car",

    #Bills
    "LA Farm Bureau": "Insurance",
    "Mint Mobile" : "Phone",

    #Entertainment/Digital
    "Fandango": "Entertainment",
    "Spotify": "Subscription",
    "Rocket Money": "Subscription",
    "Apple": "Digital Services",

    #Financing
    "Affirm": "Financing",

    #Fitness
    "Lambright LaTech Gym": "Fitness"
    
}


In [ ]:
df["Category"] = df["Merchant"].map(category_map)

.map() is a very useful pandas function as it takes every merchant and look for it inside category_map.

In [ ]:
df["Category"] = df["Category"].fillna("Other")

.fillna replaces the uncategorized transactions from NaN to "Other" so it's easier to understand and navigate

Fixing Financial Movements

In [ ]:
df.loc[
    df["Transaction_Type"] == "Money In",
    "Category"
    ] = "Money In"

df.loc[
    df["Transaction_Type"] == "Transfer Out",
    "Category"
    ] = "Transfer"

df.loc[
    df["Transaction_Type"] == "Debt Payment",
    "Category"
    ] = "Debt Payment"

In [ ]:
print(df.columns.tolist())

In [ ]:
print("Transaction_Type" in df.columns)

In [ ]:
df[
    [
        "Date",
        "Merchant",
        "Amount",
        "Transaction_Type",
        "Category"
    ]
]

In [ ]:
purchases = df[
    df["Transaction_Type"] == "Purchase"
    ].copy()

In [ ]:
category_spending = (
    purchases
    .groupby("Category")["Amount"]
    .sum()
    .abs()
    .sort_values(ascending=False)
)

category_spending

In [ ]:
category_spending = (
    purchases
    .groupby("Category")["Amount"]
    .sum()
    .abs()
    .sort_values(ascending=False)
)

category_spending

In [ ]:
df["Category"].value_counts()

In [ ]:
df[df["Category"] == "Other"][
    ["Merchant", "Description", "Amount"]
    ]

On this Stage categorized as stage 4, we created a column called Category. The purpose was to classify each merchant by what kind of spending it represents.
    

In [ ]:
purchases = df[
    df["Transaction_Type"] == "Purchase"
].copy()

SEPERATING ACTUAL PURCHASES. WHY USE .COPY()? BECAUSE WE'RE CREATING A SEPERATE DATAFRAME THAT WE'RE PROBABLY GOING TO MODIFY LATER

In [ ]:
purchases.head()

In [ ]:
purchases.shape

In [ ]:
money_in = df.loc[
    df["Transaction_Type"] == "Money In",
    "Amount"
    ].sum()

In [ ]:
total_spending = purchases["Amount"].abs().sum()

In [ ]:
debt_payments = df.loc[
    df["Transaction_Type"] == "Debt Payment",
    "Amount"
    ].abs().sum()

In [ ]:
transfers_out = df.loc[
    df["Transaction_Type"] == "Transfer Out",
    "Amount"
].abs().sum()



In [ ]:
print("Money In: $", round(money_in, 2))
print("Purchase Spending: $", round(total_spending, 2))
print("Debt Payments: $", round(debt_payments, 2))
print("Transfers Out: $", round(transfers_out, 2))

In [ ]:
net_cash_flow = df["Amount"].sum()

In [ ]:
print("Net Cash Flow: $", round(net_cash_flow, 2))

Spending by Category

In [ ]:
category_spending = (
    purchases
    .groupby("Category")["Amount"]
    .sum()
    .abs()
    .sort_values(ascending=False)
)

category_spending

In [ ]:
category_percent = (
    category_spending / total_spending
) * 100

In [ ]:
category_percent.round(2)

In Above Cell The Numerical Values Represents Part of Spending/Total Spending * 100 Generating The Total Percentage Of Spending Per Category

Combining dollars and percentages

In [ ]:
category_report = pd.DataFrame({
    "Amount": category_spending,
    "Percent": category_percent
})

category_report["Percent"] = category_report["Percent"].round(2)

category_report